# dnn_edgeiiot: Edge-IIoTset deneyi (Kişi 3, Kaggle)

Akış: repoyu çek → `src/model_dnn_edgeiiot.py`'yi çalıştır → sonuçlara bak → `outputs/dnn_edgeiiot`'yı GitHub'a gönder.

**Bir kerelik hazırlık (Kaggle):**
1. Sağ panel → **Input → + Add Input** → `Edge-IIoTset Cyber Security Dataset of IoT & IIoT` (yazar: Mohamed Amine FERRAG). Hiçbir şey indirilmez.
2. **Session options → Internet: On** (telefon doğrulaması gerekir). Hızlı deneme için Accelerator: None yeterli; asıl deneyde XGB için GPU açılabilir.
3. GitHub'da *Settings → Developer settings → Personal access tokens → Tokens (classic)* ile `repo` yetkili, süreli bir token oluştur. Kaggle'da **Add-ons → Secrets** bölümüne `GITHUB_TOKEN` adıyla ekle ve bu not defterine eriş. Token'ı asla hücreye yazma.

In [ ]:
import glob, os, subprocess
from kaggle_secrets import UserSecretsClient

GITHUB_USER = 'GamzeSerin'
REPO = 'bitirme-projesi'
BRANCH = 'dnn_edgeiiot'                       # kendi dalın; main'e PR ile birleşir
WORK_DIR = '/kaggle/working/work/dnn_edgeiiot'  # modeller + ara sonuçlar (Save Version ile kalır)
DATA_FILE = glob.glob('/kaggle/input/**/DNN-EdgeIIoT-dataset.csv', recursive=True)[0]
print(DATA_FILE)

token = UserSecretsClient().get_secret('GITHUB_TOKEN')
url = f'https://{token}@github.com/{GITHUB_USER}/{REPO}.git'

In [ ]:
os.chdir('/kaggle/working')
if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', '-q', url, REPO], check=True)
os.chdir(f'/kaggle/working/{REPO}')
!git fetch -q origin
!git checkout -q {BRANCH} 2>/dev/null || git checkout -q -b {BRANCH} origin/{BRANCH}
!git pull -q origin {BRANCH} --no-rebase
!pip install -q -r requirements.txt
!git log -1 --oneline

## 1) Hızlı deneme (~5–10 dk)
Etiket başına 3.000 satır, 3 fold. Kodun uçtan uca çalıştığını görmek için; çıktı ayrı klasöre yazılır.

In [ ]:
!python src/model_dnn_edgeiiot.py --data-file "{DATA_FILE}" --quick --drop-conflicts --work-dir {WORK_DIR}_quick --out-dir outputs_quick/dnn_edgeiiot

## 2) Asıl deney
Etiket başına en fazla 40.000 satır, 5-fold OOF, hiperparametre araması ve SHAP. Çakışan satırlar atılır (bkz. `data/dnn_edgeiiot/README.md`).
Oturum koparsa hücreyi tekrar çalıştır: biten modeller `WORK_DIR`'den okunur. Uzun sürerse **Save Version → Save & Run All** ile arka planda çalıştır.

In [ ]:
!python src/model_dnn_edgeiiot.py --data-file "{DATA_FILE}" --drop-conflicts --tune --shap --work-dir {WORK_DIR}

## 3) Sonuçlar

In [ ]:
import json, pandas as pd
OUT = 'outputs/dnn_edgeiiot'
metrics = json.load(open(f'{OUT}/metrics.json'))
manifest = json.load(open(f'{OUT}/manifest.json'))
summary = pd.DataFrame({m: {'OOF macro-F1': v['oof']['macro_f1'], 'Test macro-F1': v['test']['macro_f1'],
                            'Test accuracy': v['test']['accuracy'], 'Eğitim (s)': v['timing']['fit_seconds'],
                            'Tahmin (ms/1k)': v['timing']['predict_ms_per_1k']} for m, v in metrics.items()}).T
display(summary.sort_values('OOF macro-F1', ascending=False))
f1 = pd.DataFrame({m: {c: v['test']['per_class'][c]['f1'] for c in manifest['classes']} for m, v in metrics.items()})
display(f1.style.highlight_max(axis=1).format('{:.4f}'))
fine = pd.DataFrame({m: v['test_fine_label_recall'] for m, v in metrics.items()})
display(fine.sort_values(manifest['primary_model']).style.format('{:.3f}'))
print(json.dumps({k: v for k, v in manifest['data_prep'].items() if k.startswith('rows')}, indent=1))

## 4) DSS'i çalıştır ve sonuçları GitHub'a gönder

In [ ]:
!python main.py
!python main.py --demo dnn_edgeiiot --n 5

In [ ]:
!git config user.name 'Gamze Serin'
!git config user.email 'GITHUB_MAILIN@example.com'   # <-- GitHub'daki e-postan
!git add outputs/dnn_edgeiiot
!git commit -m 'dnn_edgeiiot: Edge-IIoTset sonuçları' || echo 'Değişiklik yok'
!git push -q origin {BRANCH}
print(f'GitHub\'da {BRANCH} → main için Pull Request aç.')